<a href="https://colab.research.google.com/github/sherf1207/AI-Projects/blob/main/Seq2Seq_English%E2%80%93French_Translator_with_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras import mixed_precision
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Attention, Concatenate
mixed_precision.set_global_policy('mixed_float16')

In [33]:
df=pd.read_csv('/content/english_french.csv')
df

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !
...,...,...
229798,Death is something that we're often discourage...,La mort est une chose qu'on nous décourage sou...
229799,Since there are usually multiple websites on a...,Puisqu'il y a de multiples sites web sur chaqu...
229800,If someone who doesn't know your background sa...,Si quelqu'un qui ne connaît pas vos antécédent...
229801,It may be impossible to get a completely error...,Il est peut-être impossible d'obtenir un Corpu...


In [34]:
english_words=df['English'].astype(str)
french_words=df['French'].astype(str)
french_sentences = "<start> " + french_words + " <end>"

In [35]:
# tokenizing the columns
eng_tokenizer = Tokenizer(filters='')
eng_tokenizer.fit_on_texts(english_words)
eng_sequences = eng_tokenizer.texts_to_sequences(english_words)
eng_vocab_size = len(eng_tokenizer.word_index) + 1

fr_tokenizer = Tokenizer(filters='')
fr_tokenizer.fit_on_texts(french_words)
fr_sequences = fr_tokenizer.texts_to_sequences(french_words)
fr_vocab_size = len(fr_tokenizer.word_index) + 1

In [36]:
max_eng_length = max(len(seq) for seq in eng_sequences)
max_fr_length = max(len(seq) for seq in fr_sequences)
max_eng_length = min(max_eng_length, 20)
max_fr_length = min(max_fr_length, 20)

In [37]:
eng_padded = pad_sequences(eng_sequences, maxlen=max_eng_length, padding='post')
fr_padded = pad_sequences(fr_sequences, maxlen=max_fr_length, padding='post')

In [39]:
embedding_dim= 128
latent_dim = 128
encoder_inputs = layers.Input(shape=(max_eng_length,))
encoder_embedding = layers.Embedding(eng_vocab_size, embedding_dim)(encoder_inputs)
encoder_outputs, state_h, state_c = layers.LSTM(latent_dim, return_sequences=True,
                                                return_state=True)(encoder_embedding)
encoder_states = [state_h, state_c]


decoder_inputs = layers.Input(shape=(max_fr_length-1,))
decoder_embedding = Embedding(fr_vocab_size, embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

In [40]:
attention_layer = Attention()
attention_out = attention_layer([decoder_outputs, encoder_outputs])

In [41]:
# Concatenate attention output with decoder output
concate = Concatenate(axis=-1)([decoder_outputs, attention_out])

In [42]:
decoder_dense = Dense(fr_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(concate)

In [43]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, 19)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 20, 128)   │  3,821,568 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 19, 128)   │  6,865,664 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 20, 128), │    131,584 │ embedding_4[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 19, 128), │    131,584 │ embedding_5[0][0… │
│                     │ (None, 128),      │            │ lstm_4[0][1],     │
│                     │ (None, 128)]      │            │ lstm_4[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_2         │ (None, 19, 128)   │          0 │ lstm_5[0][0],     │
│ (Attention)         │                   │            │ lstm_4[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 19, 256)   │          0 │ lstm_5[0][0],     │
│ (Concatenate)       │                   │            │ attention_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 19, 53638) │ 13,784,966 │ concatenate_2[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,735,366 (94.36 MB)

 Trainable params: 24,735,366 (94.36 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
decoder_input_data = fr_padded[:, :-1]
decoder_target_data = fr_padded[:, 1:]
decoder_target_data = np.expand_dims(decoder_target_data, -1)

In [45]:
model.fit(
    [eng_padded, decoder_input_data],
    decoder_target_data,
    epochs=5,
    batch_size=128)

Epoch 1/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 615s 341ms/step - accuracy: 0.7489 - loss: 1.8550
Epoch 2/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 604s 336ms/step - accuracy: 0.8089 - loss: 1.1702
Epoch 3/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 600s 334ms/step - accuracy: 0.8497 - loss: 0.8499
Epoch 4/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 599s 334ms/step - accuracy: 0.8766 - loss: 0.6467
Epoch 5/5
1796/1796 ━━━━━━━━━━━━━━━━━━━━ 598s 333ms/step - accuracy: 0.8949 - loss: 0.5123
